# LLM inference, hands-on: vLLM, SGLang, TensorRT-LLM and local hosting

Companion notebook to the *LLM Inference Engineering* talk. The slides make claims; this notebook measures them on your own GPU.

| | |
|---|---|
| **Who it's for** | Data scientists and ML engineers who want to see how inference behaves, not just read about it |
| **Engines** | [vLLM](https://docs.vllm.ai) (Parts A–C), [SGLang](https://docs.sglang.ai) (Part D), [TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) (Part E), llama.cpp and MLX (Part G) |
| **Model** | `Qwen/Qwen2.5-1.5B-Instruct`: small enough for one GPU, real enough to show every effect |
| **Hardware** | One NVIDIA GPU with 16 GB or more (L4, A10G, A100, H100). Colab L4 works for Parts A–C and F–H. |
| **Time** | 60–90 minutes for Parts A–C; every Part after that stands on its own |

Sections marked *(no GPU needed)* run on a laptop. Sections marked *(reading)* are reference material with nothing to execute. Everything else needs an NVIDIA GPU.

> **The one idea to carry through:** prefill (reading the prompt) is compute-bound and sets time to first token; decode (writing the answer) is memory-bandwidth-bound and sets tokens per second. Almost every experiment below is a variation on that.

## Table of contents

**Part A: Basics** *(single engine, offline)*
1. Setup & GPU check
2. Tokens: what the model actually sees
3. Back-of-envelope math: memory, KV cache, speed of light
4. First generation with the offline `LLM` engine
5. Sampling parameters: temperature, top-p, n, seeds, logprobs
6. Chat templates & `llm.chat`
7. Batching: sequential vs batched throughput: sequential vs batched throughput
8. Measured vs theoretical decode speed

**Part B: Intermediate** *(OpenAI-compatible server)*

9. Launch `vllm serve`
10. Streaming & measuring TTFT / TPOT yourself
11. Concurrency sweep: the latency–throughput trade-off, live
12. Prefix caching: cold vs warm TTFT
13. Reading the `/metrics` endpoint
14. Structured output (JSON schema)
15. `vllm bench serve`: the official benchmark

**Part C: Advanced**

16. KV cache capacity & engine knobs
17. Quantization: AWQ 4-bit and FP8 KV cache
18. Speculative decoding (n-gram): when it helps
19. Multi-LoRA serving
20. Scaling out: tensor parallelism & disaggregated serving
21. Tuning playbook & cheat sheet

**Part D: SGLang**

22. Install & launch an SGLang server
23. OpenAI API + native `/generate`
24. RadixAttention: `cached_tokens` and TTFT
25. The SGLang frontend: `fork` for LLM-as-judge
26. One benchmark harness for any engine

**Part E: NVIDIA's stack**

27. TensorRT-LLM offline `LLM` API
28. `trtllm-serve` + the same harness
29. Head-to-head: vLLM vs SGLang vs TRT-LLM on your GPU

**Part F: The hardware, measured**

30. Inspect your GPU, measure bandwidth & TFLOPS, find your ridge point

**Part G: Hosting big models locally**

31. A fit-and-speed planner for any Hugging Face model
32. The local toolbox: llama.cpp, MLX, Ollama, exo, KTransformers
33. vLLM when the model *almost* fits
34. Edge-case checklist
35. What practitioners report

**Part H: Kernel level, and data-scientist workloads**

36. A measured roofline
37. FlashAttention vs naive attention
38. Chunked prefill, live
39. Embeddings and zero-shot labelling at scale
40. Profiling a step

---
# Part A: Basics

## 1. Setup & GPU check

Install vLLM plus a few helpers. On Colab, pick **Runtime → Change runtime type → L4 GPU** first.

> **Watch out.** vLLM pins a specific PyTorch/CUDA build. Install it into a **fresh** environment (venv/conda/Colab) to avoid version conflicts.

In [ ]:
%pip install -q vllm openai matplotlib pandas

In [ ]:
import torch, subprocess, platform
print("Python :", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
try:
    import vllm; print("vLLM   :", vllm.__version__)
except ImportError:
    print("vLLM not installed (fine for the *(no GPU needed)* cells)")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

### Shared configuration & helpers

Run this cell **every time you restart the kernel**. Each Part only depends on it.

In [ ]:
import os, gc, time, json, statistics as st
import torch
import pandas as pd
import matplotlib.pyplot as plt

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 1.5B params, 28 layers, GQA with 2 KV heads
MAX_LEN = 4096                          # context we allow per request (smaller = more room for KV cache)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": .3})

def shutdown(llm):
    # Release the GPU memory held by an offline vLLM engine.
    # If memory still isn't freed (it happens in notebooks), restart the kernel
    # and re-run this config cell; each Part is self-contained.
    try:
        llm.llm_engine.engine_core.shutdown()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_mem():
    if not torch.cuda.is_available():
        return "no GPU"
    free, total = torch.cuda.mem_get_info()
    return f"GPU memory used: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB"

print("Config ready:", MODEL)

## 2. Tokens: what the model actually sees *(no GPU needed)*

An LLM never sees text. A **tokenizer** turns text into integer IDs from a fixed vocabulary, and every cost, limit and latency number in inference is counted in **tokens**.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

text = "Inference engineering is surprisingly fun!"
ids = tok.encode(text)
print(f"{len(text)} characters → {len(ids)} tokens   (vocab size: {tok.vocab_size:,})\n")
for i in ids:
    print(f"{i:>7}  →  {tok.decode([i])!r}")

**Try it:** tokenize code, a non-English sentence, or a long number. Notice how the characters-per-token ratio changes, and with it cost and latency.

In [ ]:
samples = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Code":    "def add(a, b):\n    return a + b",
    "Hindi":   "मशीन लर्निंग बहुत रोचक है",
    "Number":  "3.14159265358979323846",
}
for name, s in samples.items():
    n = len(tok.encode(s))
    print(f"{name:8} {len(s):3} chars → {n:3} tokens  ({len(s)/n:.1f} chars/token)")

### Chat templates

Chat models expect messages wrapped in special tokens. The **chat template** does this. It's the exact string that gets *prefilled*.

In [ ]:
messages = [{"role": "system", "content": "You are a concise assistant."},
            {"role": "user", "content": "What is a KV cache?"}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print("Prompt tokens:", len(tok.encode(prompt)))

## 3. Back-of-envelope math *(no GPU needed)*

Before touching a GPU, an inference engineer can predict most of what will happen with three formulas:

| Quantity | Formula |
|---|---|
| **Weights** | `params × bytes_per_param` |
| **KV cache per token** | `2 (K and V) × layers × kv_heads × head_dim × bytes` |
| **Decode speed ceiling** (batch 1) | `memory_bandwidth ÷ bytes_read_per_step` |

We read the real architecture from the model's config.

In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL)
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
print(f"layers={cfg.num_hidden_layers}  q_heads={cfg.num_attention_heads}  "
      f"kv_heads={cfg.num_key_value_heads}  head_dim={head_dim}")

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_elem=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_elem

MODELS = {   # name: (params in billions, layers, kv_heads, head_dim)
    "Qwen2.5-1.5B":  (1.54, cfg.num_hidden_layers, cfg.num_key_value_heads, head_dim),
    "Llama-3.1-8B":  (8.0, 32, 8, 128),
    "Llama-3.1-70B": (70.6, 80, 8, 128),
}
print(f"\n{'model':15} {'weights BF16':>13} {'KV/token':>10} {'8K-token chat':>14}")
for name, (p, L, H, D) in MODELS.items():
    kv = kv_bytes_per_token(L, H, D)
    print(f"{name:15} {p*2:>10.1f} GB {kv/1024:>7.0f} KB {kv*8192/1e9:>11.2f} GB")

### How many users fit?

With an 80 GB GPU and a 0.9 memory budget, everything not used by weights becomes KV cache. That decides the maximum concurrency.

In [ ]:
def max_concurrent(params_b, L, H, D, gpu_gb=80, util=0.9, ctx=8192, w_bytes=2, kv_bytes=2):
    free = gpu_gb * util - params_b * w_bytes          # GB left for KV cache
    per_seq = kv_bytes_per_token(L, H, D, kv_bytes) * ctx / 1e9
    return max(0, int(free / per_seq))

fig, ax = plt.subplots()
ctxs = [1024, 2048, 4096, 8192, 16384, 32768]
for name, (p, L, H, D) in MODELS.items():
    if p * 2 < 72:   # 70B BF16 doesn't fit on one 80 GB GPU at all
        ax.plot(ctxs, [max_concurrent(p, L, H, D, ctx=c) for c in ctxs], marker="o", label=f"{name} BF16")
ax.plot(ctxs, [max_concurrent(70.6, 80, 8, 128, ctx=c, w_bytes=0.5) for c in ctxs], marker="s", ls="--",
        label="Llama-3.1-70B INT4")
ax.set(xscale="log", yscale="log", xlabel="context length per sequence (tokens)",
       ylabel="max concurrent sequences", title="Concurrency that fits on one 80 GB GPU")
ax.legend(); plt.show()

> Small models with few KV heads (Qwen2.5-1.5B has only **2**) can hold *thousands* of concurrent sequences. For big models, **context length × concurrency** is what runs out first, not weights.

### The decode "speed of light"

At batch size 1, generating each token means streaming *all* the weights from HBM to the compute units once.

In [ ]:
BANDWIDTH_GBs = {"H200": 4800, "H100": 3350, "A100": 2039, "L40S": 864, "A10G": 600, "L4": 300, "T4": 320}

def decode_ceiling(params_b, bytes_per_param, bw_gbs):
    return bw_gbs / (params_b * bytes_per_param)    # tokens/s, upper bound, batch 1

for gpu in ["H100", "A100", "L4"]:
    print(f"{gpu:5}: 8B BF16 ≤ {decode_ceiling(8, 2, BANDWIDTH_GBs[gpu]):5.0f} tok/s | "
          f"8B FP8 ≤ {decode_ceiling(8, 1, BANDWIDTH_GBs[gpu]):5.0f} tok/s | "
          f"1.5B BF16 ≤ {decode_ceiling(1.54, 2, BANDWIDTH_GBs[gpu]):5.0f} tok/s")

We'll compare these predictions to measured numbers in step 8.

## 4. First generation with the offline `LLM` engine

`vllm.LLM` is the **offline / batch** API: no server, just Python. Perfect for evals, data generation and batch jobs.

While it loads, **read the logs**. They tell you a lot:

| Log line contains | What it means |
|---|---|
| `Model loading took … GiB` | Memory used by the weights |
| `GPU KV cache size: N tokens` | Total tokens of KV cache that fit, which is your capacity |
| `Maximum concurrency for 4096 tokens per request: X` | How many *full-length* requests fit at once |
| `Capturing CUDA graphs` | vLLM pre-records decode kernels to kill CPU launch overhead |

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.85)
print(gpu_mem())

In [ ]:
params = SamplingParams(temperature=0.7, max_tokens=128)
out = llm.generate(["Explain in two sentences why LLM decoding is memory-bandwidth bound."], params)[0]

print(out.outputs[0].text)
print(f"\nprompt tokens: {len(out.prompt_token_ids)} | generated tokens: {len(out.outputs[0].token_ids)} "
      f"| finish reason: {out.outputs[0].finish_reason}")

> `finish_reason` is `stop` (hit EOS / a stop string) or `length` (hit `max_tokens`). Silent truncation by `length` is one of the most common production bugs.

## 5. Sampling parameters

Each decode step produces a probability distribution over the vocabulary. **Sampling** picks one token from it.

| Parameter | Effect |
|---|---|
| `temperature` | `0` = greedy (deterministic). Higher = flatter distribution, more diverse |
| `top_p` / `top_k` / `min_p` | Cut the long tail of unlikely tokens |
| `n` | Several completions per prompt; they **share** the prompt's KV cache |
| `seed` | Reproducible sampling |
| `stop` | Stop strings |
| `logprobs` | Return the top-k probabilities at each step |

In [ ]:
prompt = "Give me a creative name for a GPU inference startup:"
for t in [0.0, 0.7, 1.5]:
    sp = SamplingParams(temperature=t, n=3, max_tokens=12, seed=42, stop=["\n"])
    outs = llm.generate([prompt], sp)[0].outputs
    print(f"temperature={t}:", [o.text.strip() for o in outs])

### Peeking at the probability distribution

`logprobs=5` returns the 5 most likely candidates at every step, so you can see exactly what the sampler chose from.

In [ ]:
import math
sp = SamplingParams(temperature=0, max_tokens=6, logprobs=5)
o = llm.generate(["The capital of France is"], sp)[0].outputs[0]
for step, (tid, cand) in enumerate(zip(o.token_ids, o.logprobs)):
    ranked = sorted(cand.items(), key=lambda kv: -kv[1].logprob)
    shown = "  ".join(f"{lp.decoded_token!r}:{math.exp(lp.logprob):.2f}" for _, lp in ranked)
    print(f"step {step}: chose {tok.decode([tid])!r:12} | {shown}")

## 6. Chat: `llm.chat` applies the template for you

In [ ]:
conversation = [
    {"role": "system", "content": "You are an expert inference engineer. Answer in 3 bullet points."},
    {"role": "user", "content": "How do I reduce time-to-first-token?"},
]
print(llm.chat(conversation, SamplingParams(temperature=0.3, max_tokens=200))[0].outputs[0].text)

## 7. Batching: sequential vs batched throughput

Decode is memory-bound: each step reads every weight once *whether it serves 1 sequence or 100*. So batching many sequences into the same step is nearly free, up to the point where compute becomes the bottleneck.

We generate the same 64 prompts two ways:
1. **One at a time** (what a naive server does)
2. **All at once** (vLLM's continuous batching)

In [ ]:
topics = ["GPUs", "KV caches", "tokenizers", "quantization", "batching", "attention", "transformers", "latency"]
prompts = [f"Write a short paragraph about {topics[i % len(topics)]} (variant {i})." for i in range(64)]
sp = SamplingParams(temperature=0.8, max_tokens=128, ignore_eos=True)   # fixed length → fair comparison

t0 = time.perf_counter()
for p in prompts[:8]:                      # only 8 sequentially, it's slow
    llm.generate([p], sp, use_tqdm=False)
seq_tps = 8 * 128 / (time.perf_counter() - t0)

t0 = time.perf_counter()
llm.generate(prompts, sp, use_tqdm=False)
batch_tps = 64 * 128 / (time.perf_counter() - t0)

print(f"sequential: {seq_tps:7.0f} output tokens/s")
print(f"batched   : {batch_tps:7.0f} output tokens/s   →  {batch_tps/seq_tps:.0f}× more throughput, same GPU")

### How far does it scale?

Sweep the batch size and watch throughput climb, then flatten as we move from memory-bound toward compute-bound (the roofline from the slides).

In [ ]:
results = []
for bs in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    batch = (prompts * 4)[:bs]
    t0 = time.perf_counter()
    llm.generate(batch, sp, use_tqdm=False)
    dt = time.perf_counter() - t0
    results.append((bs, bs * 128 / dt, 128 / dt))
    print(f"batch {bs:4}: {bs*128/dt:8.0f} tok/s total | {128/dt:6.1f} tok/s per sequence")

bs_, tot, per = zip(*results)
fig, ax1 = plt.subplots()
ax1.plot(bs_, tot, "o-", color="tab:purple", label="total throughput"); ax1.set_xscale("log", base=2)
ax1.set(xlabel="batch size", ylabel="total tokens/s")
ax2 = ax1.twinx(); ax2.plot(bs_, per, "s--", color="tab:red", label="per-sequence speed"); ax2.set_ylabel("tokens/s per sequence")
fig.legend(loc="upper center", ncol=2); plt.title("Batching: throughput vs per-user speed"); plt.show()

> Total throughput goes up roughly linearly at first while per-sequence speed barely drops. That's the memory-bound regime, where batching is free. Eventually per-sequence speed falls: the GPU is now compute-bound. **Every production deployment picks a point on this curve.**

## 8. Measured vs theoretical decode speed

Compare the batch-1 decode speed you just measured with the bandwidth ceiling from step 3.

In [ ]:
gpu_name = torch.cuda.get_device_name(0)
bw = next((v for k, v in BANDWIDTH_GBs.items() if k in gpu_name), None)
measured = results[0][2]   # batch-1 tokens/s from the sweep (includes a little prefill time)
if bw:
    ceiling = decode_ceiling(1.54, 2, bw)
    print(f"{gpu_name}: ceiling ≈ {ceiling:.0f} tok/s | measured ≈ {measured:.0f} tok/s "
          f"→ {measured/ceiling:.0%} of speed-of-light")
else:
    print(f"Unknown bandwidth for {gpu_name}; add it to BANDWIDTH_GBs. Measured: {measured:.0f} tok/s")

> For a model this small, fixed per-step overheads (kernel launches, sampling, Python) are a big share, so you'll often land well below the ceiling. For 8B+ models the measured number gets much closer to `bandwidth ÷ bytes`.

**Free the GPU before Part B:**

In [ ]:
shutdown(llm); del llm
print(gpu_mem())

---
# Part B: Intermediate: serving with `vllm serve`

In production you run vLLM as an **OpenAI-compatible HTTP server**. Any OpenAI SDK, LangChain, or `curl` works against it unchanged.

```
client ──HTTP──▶ API server (tokenize, chat template) ──▶ EngineCore (scheduler + KV cache manager) ──▶ GPU worker
       ◀──SSE stream── detokenize ◀───────────────────────────────── sampled tokens ◀────────────────────┘
```

## 9. Launch the server

We start it in the background and write logs to `vllm_server.log`. In a terminal you'd simply run:

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct --max-model-len 4096 --gpu-memory-utilization 0.85
```

In [ ]:
import subprocess, urllib.request

PORT = 8000
BASE = f"http://localhost:{PORT}"
log = open("vllm_server.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", str(PORT),
     "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.85"],
    stdout=log, stderr=subprocess.STDOUT)

def wait_healthy(timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            raise RuntimeError("server exited; see vllm_server.log")
        try:
            if urllib.request.urlopen(f"{BASE}/health", timeout=2).status == 200:
                return time.time() - t0
        except Exception:
            time.sleep(2)
    raise TimeoutError("server not healthy; see vllm_server.log")

print(f"server ready in {wait_healthy():.0f}s (that's your cold start: weights load + compile + CUDA graphs)")

In [ ]:
# The capacity lines from the startup log
for line in open("vllm_server.log"):
    if any(k in line for k in ["KV cache size", "Maximum concurrency", "Model loading took"]):
        print(line.strip()[-140:])

## 10. Streaming & measuring TTFT / TPOT yourself

| Metric | Definition |
|---|---|
| **TTFT** | time from request sent → first content token received |
| **TPOT / ITL** | average gap between subsequent tokens = `(t_last − t_first) / (N − 1)` |
| **E2E** | `TTFT + (N−1) × TPOT` |

In [ ]:
from openai import OpenAI, AsyncOpenAI
client = OpenAI(base_url=f"{BASE}/v1", api_key="not-needed")

def measure_stream(messages, max_tokens=200, **kw):
    t0 = time.perf_counter(); first = last = None; n = 0; text = []
    stream = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens,
                                            stream=True, stream_options={"include_usage": True}, **kw)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            now = time.perf_counter(); first = first or now; last = now
            text.append(chunk.choices[0].delta.content)
        if chunk.usage:
            n = chunk.usage.completion_tokens
    ttft = first - t0
    tpot = (last - first) / max(n - 1, 1)
    return {"ttft_ms": ttft * 1e3, "tpot_ms": tpot * 1e3, "tok_per_s": 1 / tpot, "tokens": n,
            "e2e_s": last - t0, "text": "".join(text)}

r = measure_stream([{"role": "user", "content": "Explain continuous batching to a new engineer."}])
print(r["text"][:400], "...\n")
print({k: round(v, 1) for k, v in r.items() if k != "text"})

## 11. Concurrency sweep: the latency–throughput trade-off, live

We fire N simultaneous requests and record per-request TTFT and TPOT plus total system throughput. This is the same curve from the slides, measured on your GPU.

In [ ]:
import asyncio
aclient = AsyncOpenAI(base_url=f"{BASE}/v1", api_key="not-needed")

async def one_request(i, max_tokens=256):
    t0 = time.perf_counter(); first = last = None; n = 0
    stream = await aclient.chat.completions.create(
        model=MODEL, max_tokens=max_tokens, stream=True, stream_options={"include_usage": True},
        extra_body={"ignore_eos": True},   # fixed output length → comparable numbers
        messages=[{"role": "user", "content": f"Write a story about GPU number {i}."}])
    async for c in stream:
        if c.choices and c.choices[0].delta.content:
            now = time.perf_counter(); first = first or now; last = now
        if c.usage: n = c.usage.completion_tokens
    return first - t0, (last - first) / max(n - 1, 1), n

async def sweep(levels=(1, 4, 16, 64, 128)):
    rows = []
    for c in levels:
        t0 = time.perf_counter()
        res = await asyncio.gather(*[one_request(i) for i in range(c)])
        wall = time.perf_counter() - t0
        ttfts, tpots, ns = zip(*res)
        rows.append({"concurrency": c, "ttft_p50_ms": st.median(ttfts) * 1e3,
                     "ttft_max_ms": max(ttfts) * 1e3, "tpot_p50_ms": st.median(tpots) * 1e3,
                     "per_user_tok_s": 1 / st.median(tpots), "system_tok_s": sum(ns) / wall})
        print(rows[-1])
    return rows

rows = await sweep()

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).round(1); display(df)

fig, ax = plt.subplots()
ax.plot(df.system_tok_s, df.per_user_tok_s, "o-")
for _, r in df.iterrows():
    ax.annotate(f"c={int(r.concurrency)}", (r.system_tok_s, r.per_user_tok_s), textcoords="offset points", xytext=(6, 4))
ax.set(xlabel="system throughput (tokens/s)", ylabel="per-user speed (tokens/s)",
       title="Every deployment picks a point on this curve")
plt.show()

> Read the table like an SLO owner: *"We need ≥ 30 tok/s per user and TTFT P50 < 500 ms. What's the highest concurrency that still meets that?"* The system throughput **at that point** is your **goodput**, and it sets how many GPUs you need.

## 12. Prefix caching: cold vs warm TTFT

vLLM hashes KV-cache blocks by their content plus everything before them. A request that starts with an already-seen prefix **skips prefill** for that part. It's on by default.

We build a ~3,000-token system prompt (think: a long policy doc, tool definitions or RAG context) and compare:
- **cold**: first time this prefix is seen
- **warm**: same prefix, *different* question
- **control**: a different prefix, cold again

In [ ]:
import uuid
def long_system_prompt(tag):
    clauses = [f"Clause {i}: requests of category {i % 7} must be answered within {i % 5 + 1} business days "
               f"and escalated to tier {i % 3 + 1} support if unresolved." for i in range(110)]
    return f"[doc {tag}] You are a support assistant. Company policy:\n" + "\n".join(clauses)

doc_a = long_system_prompt(uuid.uuid4().hex)
doc_b = long_system_prompt(uuid.uuid4().hex)
print("system prompt tokens:", len(tok.encode(doc_a)) if "tok" in globals() else "~3000")

def ttft(system, question):
    return measure_stream([{"role": "system", "content": system}, {"role": "user", "content": question}],
                          max_tokens=1)["ttft_ms"]

cold    = ttft(doc_a, "What does clause 12 say?")
warm    = ttft(doc_a, "Summarize clause 40.")
control = ttft(doc_b, "What does clause 12 say?")
print(f"cold    : {cold:6.1f} ms\nwarm    : {warm:6.1f} ms   ← prefix reused\ncontrol : {control:6.1f} ms")

> **Prompt design rule:** static content first (system prompt, tools, documents), dynamic content last. Putting a timestamp or user ID at the *top* of the prompt silently destroys your cache hit rate.

> **Watch out.** `tok` comes from Part A. If you restarted the kernel, the token count line just prints an estimate.

## 13. Reading the `/metrics` endpoint

vLLM exposes Prometheus metrics. These are what you'd put on a Grafana dashboard and autoscale on.

In [ ]:
import re
raw = urllib.request.urlopen(f"{BASE}/metrics").read().decode()
KEYS = ["num_requests_running", "num_requests_waiting", "kv_cache_usage", "cache_usage",
        "prefix_cache_queries", "prefix_cache_hits", "prompt_tokens_total", "generation_tokens_total",
        "time_to_first_token_seconds_sum", "time_to_first_token_seconds_count", "num_preemptions"]
for line in raw.splitlines():
    if line.startswith("vllm:") and any(k in line for k in KEYS):
        name = re.sub(r"\{.*\}", "", line.split()[0]); print(f"{name:55} {float(line.split()[-1]):,.3f}")

| Metric | Why you watch it |
|---|---|
| `num_requests_waiting` | Queueing: **the** autoscaling signal |
| `kv_cache_usage_perc` | Near 100% → preemptions and latency spikes |
| `prefix_cache_hits / queries` | Hit rate; low means wasted prefill |
| `time_to_first_token_seconds` | TTFT histogram → P50/P99 |
| `num_preemptions` | Requests evicted for lack of KV memory |

Metric names can change slightly between vLLM versions; the filter above matches by substring.

## 14. Structured output (JSON schema)

vLLM can **guarantee** schema-valid JSON by masking invalid tokens at every decode step (grammar-constrained decoding). Use it for extraction, tool calls and agents.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "gpu": {"type": "string"},
        "memory_gb": {"type": "integer"},
        "bandwidth_tb_s": {"type": "number"},
        "good_for": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
    },
    "required": ["gpu", "memory_gb", "bandwidth_tb_s", "good_for"],
}
resp = client.chat.completions.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "Describe the NVIDIA H100 SXM GPU for LLM inference."}],
    response_format={"type": "json_schema", "json_schema": {"name": "gpu_card", "schema": schema}})
print(json.dumps(json.loads(resp.choices[0].message.content), indent=2))

> **Watch out.** The **format** is guaranteed, the **facts** are not. A 1.5B model may get specs wrong. Constrained decoding fixes shape, not knowledge.

## 15. `vllm bench serve`: the official load generator

For real capacity planning, use the built-in benchmark with **your** prompt/output length distribution. Here: random 512-token prompts, 128-token outputs, 32 concurrent users.

In [ ]:
!vllm bench serve --model {MODEL} --port {PORT} \
    --dataset-name random --random-input-len 512 --random-output-len 128 \
    --num-prompts 200 --max-concurrency 32 2>&1 | tail -n 30

**Try it:** Re-run with `--max-concurrency 1, 8, 64` and with `--random-input-len 4000`. Which metric moves most when prompts get long? (Hint: prefill → TTFT.)

**Stop the server before Part C:**

In [ ]:
server.terminate(); server.wait(timeout=60); log.close()
time.sleep(5); print("server stopped |", gpu_mem())

---
# Part C: Advanced

Each section below starts its own engine. If you get out-of-memory errors, **restart the kernel and re-run the config cell** from step 1.

## 16. KV cache capacity & engine knobs

| Flag (`LLM(...)` kwarg) | Controls | Trade-off |
|---|---|---|
| `gpu_memory_utilization` | Share of VRAM vLLM may use | More → bigger KV cache → more concurrency |
| `max_model_len` | Max context per request | Lower → more sequences fit |
| `max_num_seqs` | Max sequences per step | Higher → throughput; lower → per-user speed |
| `max_num_batched_tokens` | Token budget per step (chunked prefill) | Smaller → smoother TPOT; larger → faster TTFT |
| `kv_cache_dtype="fp8"` | KV cache precision | ~2× KV capacity, tiny quality cost |
| `enable_prefix_caching` | Prefix reuse (default on) | Free win when prompts share prefixes |

### Experiment: how `gpu_memory_utilization` changes capacity

Watch the `GPU KV cache size` and `Maximum concurrency` log lines for each run.

In [ ]:
from vllm import LLM, SamplingParams

def kv_capacity(**kw):
    llm = LLM(model=MODEL, max_model_len=MAX_LEN, **kw)
    blocks = None
    try:   # not exposed identically in every version; the startup log is the source of truth
        blocks = llm.llm_engine.vllm_config.cache_config.num_gpu_blocks
    except Exception:
        pass
    shutdown(llm); del llm
    return blocks

for util in [0.4, 0.8]:
    b = kv_capacity(gpu_memory_utilization=util)
    print(f"\n>>> gpu_memory_utilization={util}: num_gpu_blocks={b} "
          f"(× 16 tokens/block ≈ {b*16 if b else '?'} tokens)\n")

> Weights are a fixed cost. **Every extra GB you give vLLM goes to KV cache**, i.e. straight into concurrency.

## 17. Quantization

| Variant | What's quantized | Helps |
|---|---|---|
| **AWQ / GPTQ INT4** (W4A16) | weights → 4-bit | memory-bound decode, small batches; 4× smaller weights |
| **FP8** (W8A8) | weights + activations → 8-bit | compute-bound prefill & big batches (Ada/Hopper+) |
| **FP8 KV cache** | KV cache → 8-bit | ~2× more tokens in the cache |

We compare the BF16 model with its official AWQ 4-bit checkpoint on single-stream latency and batch throughput, and eyeball output quality.

In [ ]:
def profile(model, label, **kw):
    llm = LLM(model=model, max_model_len=MAX_LEN, gpu_memory_utilization=0.6, **kw)
    sp = SamplingParams(temperature=0, max_tokens=128, ignore_eos=True)
    llm.generate(["warmup"], sp, use_tqdm=False)
    t0 = time.perf_counter(); llm.generate(["Explain the roofline model."], sp, use_tqdm=False)
    single = 128 / (time.perf_counter() - t0)
    batch = [f"Explain topic {i} in machine learning." for i in range(128)]
    t0 = time.perf_counter(); llm.generate(batch, sp, use_tqdm=False)
    thr = 128 * 128 / (time.perf_counter() - t0)
    sample = llm.generate(["In one sentence, what is a KV cache?"],
                          SamplingParams(temperature=0, max_tokens=60), use_tqdm=False)[0].outputs[0].text
    shutdown(llm); del llm
    return {"variant": label, "single_stream_tok_s": round(single), "batch128_tok_s": round(thr), "sample": sample.strip()}

quant_rows = [profile(MODEL, "BF16"),
              profile("Qwen/Qwen2.5-1.5B-Instruct-AWQ", "AWQ INT4")]
display(pd.DataFrame(quant_rows))

In [ ]:
# FP8 KV cache: same weights, half-size cache. Compare the "GPU KV cache size" log line with BF16's.
# Needs a GPU/backend with FP8 KV support (Ada / Hopper / Blackwell recommended).
try:
    quant_rows.append(profile(MODEL, "BF16 + FP8 KV cache", kv_cache_dtype="fp8"))
    display(pd.DataFrame(quant_rows))
except Exception as e:
    print("FP8 KV cache not supported here:", e)

> On a model this small, weights are already tiny, so INT4 gains are modest and kernel overheads dominate. On 70B-class models the same switch is the difference between "needs 2 GPUs" and "fits on 1".
>
> **Watch out.** Always run **your own task evals** before shipping a quantized model. Perplexity alone hides regressions.

## 18. Speculative decoding (n-gram)

**Idea:** decode leaves compute idle, so guess *k* tokens cheaply and let the big model verify all of them in **one** forward pass. The output distribution is unchanged.

**N-gram / prompt lookup** needs no extra model: it proposes continuations by copying spans that already appear in the prompt. Ideal for **code editing, rewriting, RAG and summarization**, where the output repeats the input.

```python
speculative_config = {"method": "ngram", "num_speculative_tokens": 4,
                      "prompt_lookup_min": 2, "prompt_lookup_max": 4}
```

In [ ]:
CODE = '''
def compute_stats(values):
    total = 0
    count = 0
    for v in values:
        total = total + v
        count = count + 1
    mean = total / count
    variance = 0
    for v in values:
        variance = variance + (v - mean) ** 2
    variance = variance / count
    return {"mean": mean, "variance": variance, "count": count}
'''
edit_prompt = [{"role": "user", "content":
    "Rename the variable `total` to `running_sum` in this code. Return the full code only.\n" + CODE}]

def time_single(llm, runs=3):
    sp = SamplingParams(temperature=0, max_tokens=256)
    llm.chat(edit_prompt, sp, use_tqdm=False)             # warmup
    ts, n = [], 0
    for _ in range(runs):
        t0 = time.perf_counter(); o = llm.chat(edit_prompt, sp, use_tqdm=False)[0].outputs[0]
        ts.append(time.perf_counter() - t0); n = len(o.token_ids)
    return n / st.median(ts), o.text

base = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
base_tps, base_txt = time_single(base); shutdown(base); del base

spec = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6,
           speculative_config={"method": "ngram", "num_speculative_tokens": 4,
                               "prompt_lookup_min": 2, "prompt_lookup_max": 4})
spec_tps, spec_txt = time_single(spec); shutdown(spec); del spec

print(f"baseline   : {base_tps:6.0f} tok/s\nn-gram spec: {spec_tps:6.0f} tok/s  →  {spec_tps/base_tps:.2f}× speedup")
print("identical output (greedy):", base_txt == spec_txt)

**Try it:** replace the code-editing prompt with an open-ended one ("write a poem about GPUs"). Nothing in the prompt can be copied, the acceptance rate collapses, and the speedup disappears. Spec decode is **workload-dependent**.

| Method | Drafter | When |
|---|---|---|
| `ngram` | copy from prompt | edits, RAG, summarization |
| `draft_model` | small model, same tokenizer | general |
| `eagle3` | tiny head on target's hidden states | best general acceptance; needs a trained head |
| `mtp` | multi-token-prediction layers built into the model | models trained with MTP (e.g. DeepSeek-V3) |

> **Watch out.** Spec decode shines at **low concurrency**. Under heavy load the GPU is already busy and gains shrink or turn negative. Benchmark at your real traffic level.

## 19. Multi-LoRA serving

Serve many fine-tuned adapters on **one** copy of the base model. Each request picks its adapter; requests for different adapters are batched together.

```bash
# server form
vllm serve Qwen/Qwen2.5-1.5B-Instruct --enable-lora \
    --lora-modules support=/path/to/support_adapter sql=/path/to/sql_adapter
# clients just set model="support" or model="sql"
```

Set `LORA_PATH` to a local or Hugging Face LoRA adapter trained on this base model to run the offline version:

In [ ]:
LORA_PATH = None   # e.g. a HF repo id or local folder of a LoRA adapter for Qwen2.5-1.5B-Instruct

if LORA_PATH:
    from vllm.lora.request import LoRARequest
    from huggingface_hub import snapshot_download
    path = LORA_PATH if os.path.isdir(LORA_PATH) else snapshot_download(LORA_PATH)
    llm = LLM(model=MODEL, enable_lora=True, max_lora_rank=64, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
    sp = SamplingParams(temperature=0, max_tokens=80)
    q = [{"role": "user", "content": "Introduce yourself."}]
    print("base   :", llm.chat(q, sp)[0].outputs[0].text)
    print("adapter:", llm.chat(q, sp, lora_request=LoRARequest("my_adapter", 1, path))[0].outputs[0].text)
    shutdown(llm); del llm
else:
    print("Set LORA_PATH to try multi-LoRA serving.")

## 20. Scaling out: beyond one GPU *(reading)*

These need multiple GPUs, so they're shown as commands, not run.

### Tensor / pipeline / data / expert parallelism
```bash
# 70B model on 4 GPUs in one node (NVLink): split every layer → lower latency
vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 4

# 2 nodes × 8 GPUs: TP inside the node, PP across nodes
vllm serve <model> --tensor-parallel-size 8 --pipeline-parallel-size 2

# Throughput: independent replicas (or put N servers behind a load balancer)
vllm serve <model> --data-parallel-size 4

# Large MoE (DeepSeek / Qwen-MoE): spread experts across GPUs
vllm serve <moe-model> --tensor-parallel-size 8 --enable-expert-parallel
```

| Strategy | Splits | Good for | Watch out |
|---|---|---|---|
| TP | each layer's matrices | latency, models that don't fit | all-reduce every layer → needs NVLink |
| PP | layers into stages | crossing nodes | doesn't cut single-request latency |
| DP | whole replicas | throughput | memory per replica |
| EP | MoE experts | huge MoE models | all-to-all traffic, expert load balance |

### Disaggregated prefill / decode
Prefill (compute-bound) and decode (bandwidth-bound) run on **separate GPU pools**; the KV cache is shipped between them over NVLink/RDMA (e.g. NIXL). vLLM supports this via KV-transfer connectors, and orchestration layers such as **NVIDIA Dynamo** and **llm-d** manage it on Kubernetes, adding **KV-cache-aware routing** (send each request to the replica that already holds its prefix).

```
          ┌──────────── prefill pool (TTFT) ────────────┐        ┌────── decode pool (TPOT) ──────┐
request ─▶│ GPU  GPU  GPU  GPU   → builds KV cache        │──KV──▶ │ GPU  GPU  GPU  GPU → tokens    │─▶ stream
          └──────────────────────────────────────────────┘ NIXL   └────────────────────────────────┘
```
> Disaggregation pays off at scale with long prompts and strict SLOs on both TTFT and TPOT. For small deployments, **chunked prefill** (on by default) gets you most of the benefit for free.

## 21. Tuning playbook & cheat sheet *(reading)*

### Symptom → lever

| Symptom | Try (roughly in order) |
|---|---|
| **TTFT too high** | prefix caching + static-first prompts → shorter prompts → tune `max_num_batched_tokens` → more replicas / TP → disaggregate prefill |
| **TPOT too slow** | FP8 / INT4 weights → speculative decoding → TP → lower `max_num_seqs` → GPU with more bandwidth |
| **Throughput too low** | raise `max_num_seqs` → more KV room (`gpu_memory_utilization`, FP8 KV) → quantize → DP replicas |
| **OOM / preemptions** | lower `max_model_len` → FP8 KV cache → fewer concurrent seqs → TP / bigger GPU |
| **Cost too high** | smaller or distilled model → quantize → right-size GPU → batch offline jobs → raise utilization |

### The method
1. **Define the SLO** (e.g. TTFT P99 < 1 s, ≥ 30 tok/s per user).
2. **Load-test realistic traffic**: real prompt and output length distributions, real concurrency.
3. **Find the bottleneck**: prefill (TTFT) or decode (TPOT)? Memory (KV usage, preemptions) or compute?
4. **Change one knob**, re-measure, keep what helps.

### Cost formula
$$\text{\$ per 1M tokens} = \frac{\text{GPU \$/hour}}{\text{tokens/s} \times 3600} \times 10^6$$

In [ ]:
def cost_per_million(gpu_dollars_per_hour, tokens_per_second):
    return gpu_dollars_per_hour / (tokens_per_second * 3600) * 1e6

# Illustrative prices; plug in your own GPU price and the throughput you measured in step 11 / 15
for price, tps in [(3.0, 500), (3.0, 2000), (3.0, 8000)]:
    print(f"${price}/hr GPU at {tps:>5} tok/s  →  ${cost_per_million(price, tps):.2f} per 1M output tokens")

---
# Part D: SGLang, the other big open engine

SGLang (LMSYS) implements the same toolkit as vLLM (continuous batching, paged KV, chunked prefill, CUDA graphs, spec decode, TP/DP/EP, PD disaggregation). Where it differs:

| | vLLM | SGLang |
|---|---|---|
| Prefix reuse | Hash-based automatic prefix caching (block level) | **RadixAttention**: a radix tree over all cached tokens, with LRU eviction and a prefix-aware scheduler (`--schedule-policy lpm`) |
| Programming model | OpenAI API + offline `LLM` | OpenAI API + offline `sgl.Engine` + a **frontend language** (`fork`, `gen`, `select`) for multi-call programs |
| Big MoE | EP, DP attention | EP, **DP attention** (`--enable-dp-attention`), DeepEP; many DeepSeek/Kimi day-0 recipes come from here |
| Defaults to know | `--gpu-memory-utilization 0.9` | `--mem-fraction-static` (default depends on GPU) |

Modal's LLM Engine Almanac found the two **perform comparably out of the box**. Choose on features, model support and your traffic shape, and benchmark with your own prompts (§26 does exactly that).

> **Watch out.** **Environment edge case.** vLLM, SGLang and TensorRT-LLM each pin their own PyTorch/CUDA versions. Installing all three in one environment usually breaks at least one of them. Use **one virtualenv (or container) per engine**. The cells below launch each engine as a *subprocess*, so this notebook's kernel only needs `openai`, `requests` and `pandas`. Point `SGLANG_PY` at the Python of your SGLang env.

## 22. Install & launch an SGLang server

```bash
# separate env, e.g.
uv venv ~/.venvs/sglang && source ~/.venvs/sglang/bin/activate
uv pip install "sglang[all]"          # check docs.sglang.ai for the CUDA-specific wheel if this fails
```

In [ ]:
import os, sys, subprocess, time, urllib.request, json

# Free the GPU from earlier parts: two engines on one GPU will fight over memory
try:
    if server.poll() is None: server.terminate(); server.wait(60)
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

SGLANG_PY = os.environ.get("SGLANG_PY", sys.executable)   # e.g. ~/.venvs/sglang/bin/python
SG_PORT = 30000
SG_BASE = f"http://localhost:{SG_PORT}"

def launch(cmd, logfile, health_url, timeout=900):
    # Start an engine as a subprocess and block until /health answers.
    log = open(logfile, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"engine exited; tail {logfile}")
        try:
            if urllib.request.urlopen(health_url, timeout=2).status == 200:
                print(f"ready in {time.time() - t0:.0f}s"); return proc
        except Exception:
            time.sleep(2)
    proc.terminate(); raise TimeoutError(f"not healthy; see {logfile}")

sg = launch([SGLANG_PY, "-m", "sglang.launch_server",
             "--model-path", MODEL, "--port", str(SG_PORT),
             "--context-length", str(MAX_LEN),
             "--mem-fraction-static", "0.80",
             "--enable-metrics"],
            "sglang_server.log", f"{SG_BASE}/health")

In [ ]:
# The capacity lines SGLang prints at startup (wording varies a little by version)
for line in open("sglang_server.log"):
    if any(k in line for k in ["max_total_num_tokens", "KV Cache is allocated", "max_running_requests", "Load weight end"]):
        print(line.strip()[-160:])

> `max_total_num_tokens` is SGLang's equivalent of vLLM's "KV cache size in tokens". Divide by your typical context length to get a rough max concurrency, same arithmetic as §3.

## 23. Same OpenAI API, plus SGLang's native `/generate`

The OpenAI-compatible endpoint means **everything from Part B works unchanged**: just swap the base URL. The native `/generate` endpoint adds `meta_info`, which reports `cached_tokens`: how many prompt tokens were served from the radix cache.

In [ ]:
from openai import OpenAI
sg_client = OpenAI(base_url=f"{SG_BASE}/v1", api_key="none")
r = sg_client.chat.completions.create(model=MODEL, max_tokens=60,
        messages=[{"role": "user", "content": "One sentence: what is RadixAttention?"}])
print(r.choices[0].message.content)

import requests
def sg_generate(text, max_new_tokens=32):
    out = requests.post(f"{SG_BASE}/generate", json={
        "text": text, "sampling_params": {"max_new_tokens": max_new_tokens, "temperature": 0}}).json()
    return out["text"], out["meta_info"]

_, meta = sg_generate("Hello! " * 5)
print({k: meta[k] for k in ("prompt_tokens", "completion_tokens", "cached_tokens") if k in meta})

## 24. RadixAttention: watch `cached_tokens` and TTFT

Same experiment as §12, but now the engine *tells* us how much it reused. We build a ~3k-token shared prefix (think: a long system prompt or a RAG document), then send different questions behind it.

In [ ]:
DOC = " ".join(f"Clause {i}: the supplier shall deliver item {i} within {i % 30 + 1} days." for i in range(220))
QUESTIONS = ["Which clause covers item 17?", "Summarise clause 3.", "What is the longest delivery time?",
             "List clauses with 5-day delivery.", "Is item 200 covered?"]

def timed(text):
    t0 = time.perf_counter(); _, m = sg_generate(text, max_new_tokens=1)   # 1 token ≈ pure prefill time
    return (time.perf_counter() - t0) * 1e3, m.get("cached_tokens", 0), m["prompt_tokens"]

requests.post(f"{SG_BASE}/flush_cache")           # start cold
rows = []
for i, q in enumerate(QUESTIONS):
    ms, cached, total = timed(f"{DOC}\n\nQuestion: {q}\nAnswer:")
    rows.append({"request": i, "prompt_tokens": total, "cached_tokens": cached, "ttft_ms": round(ms, 1)})
# Control: same length, but the dynamic part is FIRST, so nothing can be reused
ms, cached, total = timed(f"Request id {time.time()}.\n{DOC}\n\nQuestion: {QUESTIONS[0]}\nAnswer:")
rows.append({"request": "dynamic-first", "prompt_tokens": total, "cached_tokens": cached, "ttft_ms": round(ms, 1)})
display(pd.DataFrame(rows))

> Request 0 pays for the whole prefix; requests 1–4 hit the radix tree and only prefill the question. The `dynamic-first` row is the classic mistake: a timestamp or request ID at the top of the prompt makes the cache useless. **Static first, dynamic last.**

## 25. The SGLang frontend: `fork` for parallel sub-calls

For multi-step LLM programs (judge with several criteria, tree-of-thought, agents), SGLang's frontend lets you write the program once. `fork` branches share the prefix's KV cache automatically, so you pay for the shared context once.

In [ ]:
try:
    import sglang as sgl
    sgl.set_default_backend(sgl.RuntimeEndpoint(SG_BASE))

    @sgl.function
    def judge(s, answer):
        s += sgl.system("You are a strict evaluator. Reply with a score 1-5 and one short reason.")
        s += sgl.user(f"Answer to evaluate:\n{answer}")
        forks = s.fork(3)
        for f, crit in zip(forks, ["factual accuracy", "clarity", "conciseness"]):
            f += sgl.user(f"Score it on {crit}.")
            f += sgl.assistant(sgl.gen("score", max_tokens=40, temperature=0))
        s["scores"] = [f["score"] for f in forks]

    st_ = judge.run(answer="PagedAttention stores the KV cache in fixed-size blocks, like OS virtual memory pages.")
    for c, sc in zip(["accuracy", "clarity", "concise"], st_["scores"]): print(f"{c:9} → {sc.strip()}")
except ImportError:
    print("The frontend needs `sglang` importable in THIS kernel. Skip, or run this cell from the SGLang env.")

> For data scientists this is the LLM-as-judge pattern: one shared (expensive) context, several cheap criteria. Same idea works in vLLM via prefix caching, but here it's explicit.

### SGLang flags worth knowing

| Flag | What it does | vLLM equivalent |
|---|---|---|
| `--tp N` / `--dp N` | tensor / data parallel | `--tensor-parallel-size` / `--data-parallel-size` |
| `--mem-fraction-static 0.85` | share of VRAM for weights + KV pool | `--gpu-memory-utilization` |
| `--context-length` | max context | `--max-model-len` |
| `--max-running-requests` | concurrency cap | `--max-num-seqs` |
| `--chunked-prefill-size 4096` | prefill chunk per step | `--max-num-batched-tokens` |
| `--quantization fp8` · `--kv-cache-dtype fp8_e5m2` | weights / KV precision | same names |
| `--speculative-algorithm EAGLE3 --speculative-draft-model-path …` | EAGLE spec decoding | `--speculative-config` |
| `--enable-dp-attention` | DP for attention + EP for experts (DeepSeek-style MoE) | `--data-parallel-size` + `--enable-expert-parallel` |
| `--disable-radix-cache` | turn prefix reuse off (A/B tests) | `--no-enable-prefix-caching` |
| `--disaggregation-mode prefill\|decode` | PD disaggregation | KV connector config |

Always confirm with `python -m sglang.launch_server --help` for your version; flags move between releases.

## 26. One harness, any engine

Every engine here speaks the OpenAI API, so a single async load generator gives an apples-to-apples comparison. We reuse it for TensorRT-LLM in Part E.

In [ ]:
import asyncio
from openai import AsyncOpenAI

async def bench_openai(base, model, levels=(1, 8, 32, 64), max_tokens=256, prompt_words=200):
    ac = AsyncOpenAI(base_url=f"{base}/v1", api_key="none")
    filler = " ".join(["data"] * prompt_words)
    async def one(i):
        t0 = time.perf_counter(); first = last = None; n = 0
        stream = await ac.chat.completions.create(model=model, max_tokens=max_tokens, stream=True,
            stream_options={"include_usage": True}, extra_body={"ignore_eos": True},
            messages=[{"role": "user", "content": f"[{i}] {filler}\nWrite a long story about GPU {i}."}])
        async for c in stream:
            if c.choices and c.choices[0].delta.content:
                now = time.perf_counter(); first = first or now; last = now
            if c.usage: n = c.usage.completion_tokens
        return first - t0, (last - first) / max(n - 1, 1), n
    rows = []
    for c in levels:
        await asyncio.gather(*[one(i) for i in range(min(c, 4))])          # warm-up
        t0 = time.perf_counter(); res = await asyncio.gather(*[one(i) for i in range(c)])
        wall = time.perf_counter() - t0; ttft, tpot, ns = zip(*res)
        rows.append({"concurrency": c, "ttft_p50_ms": st.median(ttft) * 1e3, "tpot_p50_ms": st.median(tpot) * 1e3,
                     "per_user_tok_s": 1 / st.median(tpot), "system_tok_s": sum(ns) / wall})
    return pd.DataFrame(rows).round(1)

results = {}
results["sglang"] = await bench_openai(SG_BASE, MODEL)
display(results["sglang"])

In [ ]:
# Official load generator. It can also target vLLM / TRT-LLM servers via --backend / --base-url.
!{SGLANG_PY} -m sglang.bench_serving --backend sglang --port {SG_PORT} \
    --dataset-name random --random-input-len 512 --random-output-len 256 \
    --num-prompts 200 --max-concurrency 32 2>&1 | tail -25

In [ ]:
sg.terminate(); sg.wait(60); print("SGLang stopped")

---
# Part E: NVIDIA's stack: TensorRT-LLM, Dynamo, NIM

| Piece | What it is | When you'd touch it |
|---|---|---|
| **TensorRT-LLM** | NVIDIA's engine. Since 1.0 the default backend is **PyTorch** (no separate engine-build step); the older TensorRT backend compiles a model-specific engine. Ships `LLM` API, `trtllm-serve`, `trtllm-bench` | Squeezing the last 10–30% out of NVIDIA GPUs, FP8/NVFP4 on Hopper/Blackwell |
| **Dynamo** | Datacenter-scale orchestration *on top of* vLLM, SGLang or TRT-LLM: PD disaggregation, KV-aware routing, NIXL KV transfer, KV offload | Multi-node serving, disaggregation |
| **NIM** | Pre-packaged containers (engine + tuned config per GPU) behind an OpenAI API | You want a supported, pre-tuned endpoint and have an NVIDIA AI Enterprise licence |
| **Model Optimizer** (`modelopt`) | Quantization toolkit (FP8, NVFP4, INT4 AWQ) producing checkpoints all three engines can load | Making your own FP8/FP4 checkpoint |

> **Watch out.** **Install edge case.** TRT-LLM is Linux + NVIDIA only and is pinned to specific CUDA/PyTorch versions. The path of least pain is the NGC container:
> ```bash
> docker run --rm -it --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 -p 8001:8001 \
>   -v ~/.cache/huggingface:/root/.cache/huggingface nvcr.io/nvidia/tensorrt-llm/release:<tag>
> ```
> `--ipc=host` (or a large `--shm-size`) matters for every engine in Docker: multi-GPU runs crash with cryptic NCCL/shared-memory errors without it.

## 27. TensorRT-LLM offline `LLM` API

Looks almost identical to vLLM's. Run this from inside the TRT-LLM container/env (it imports `tensorrt_llm`).

In [ ]:
TRT_SNIPPET = r'''
from tensorrt_llm import LLM, SamplingParams
from tensorrt_llm.llmapi import KvCacheConfig
import time

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct",
          kv_cache_config=KvCacheConfig(free_gpu_memory_fraction=0.8, enable_block_reuse=True))  # block reuse = prefix caching
prompts = [f"Write a haiku about GPU number {i}." for i in range(256)]
t0 = time.perf_counter()
outs = llm.generate(prompts, SamplingParams(max_tokens=64, temperature=0.8))
dt = time.perf_counter() - t0
n = sum(len(o.outputs[0].token_ids) for o in outs)
print(outs[0].outputs[0].text.strip())
print(f"{len(prompts)} prompts, {n} tokens in {dt:.1f}s -> {n/dt:.0f} tok/s")
'''
open("trtllm_offline.py", "w").write(TRT_SNIPPET)
TRTLLM_PY = os.environ.get("TRTLLM_PY")       # python inside the TRT-LLM env, if you have one
if TRTLLM_PY:
    print(subprocess.run([TRTLLM_PY, "trtllm_offline.py"], capture_output=True, text=True).stdout[-1500:])
else:
    print("Wrote trtllm_offline.py. Run it inside the TRT-LLM container:  python trtllm_offline.py")

## 28. `trtllm-serve` + the same benchmark harness

`trtllm-serve` exposes the same OpenAI API, so §26's `bench_openai` works unchanged. Start it inside the container (or set `TRTLLM_SERVE` to the binary path):

```bash
trtllm-serve Qwen/Qwen2.5-1.5B-Instruct --host 0.0.0.0 --port 8001 \
    --max_batch_size 128 --max_num_tokens 8192 --max_seq_len 4096 \
    --kv_cache_free_gpu_memory_fraction 0.8
# extra knobs (spec decode, CUDA graphs, attention backend…) go in a YAML file:
#   --extra_llm_api_options extra.yaml
```

In [ ]:
TRT_BASE = "http://localhost:8001"
trt = None
if os.environ.get("TRTLLM_SERVE"):
    trt = launch([os.environ["TRTLLM_SERVE"], MODEL, "--port", "8001", "--max_seq_len", str(MAX_LEN),
                  "--kv_cache_free_gpu_memory_fraction", "0.8"], "trtllm_server.log", f"{TRT_BASE}/health")
try:
    urllib.request.urlopen(f"{TRT_BASE}/health", timeout=2)
    results["trtllm"] = await bench_openai(TRT_BASE, MODEL); display(results["trtllm"])
except Exception as e:
    print("No TRT-LLM server on :8001, skipping.", type(e).__name__)

## 29. Head-to-head on your GPU

Re-launch vLLM with matching memory settings and benchmark it with the **same** harness, then plot all engines on the latency–throughput plane.

In [ ]:
server = launch(["vllm", "serve", MODEL, "--port", str(PORT), "--max-model-len", str(MAX_LEN),
                 "--gpu-memory-utilization", "0.80"], "vllm_server.log", f"{BASE}/health")
results["vllm"] = await bench_openai(BASE, MODEL)
server.terminate(); server.wait(60)

fig, ax = plt.subplots()
for name, df_ in results.items():
    ax.plot(df_.system_tok_s, df_.per_user_tok_s, "o-", label=name)
    for _, r in df_.iterrows():
        ax.annotate(int(r.concurrency), (r.system_tok_s, r.per_user_tok_s), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set(xlabel="system throughput (tok/s)", ylabel="per-user speed (tok/s)", title="Same model, same GPU, same harness")
ax.legend(); plt.show()
display(pd.concat(results, names=["engine"]).reset_index(level=0))

> **Watch out.** **How to read an engine comparison honestly**
> - Same model weights, same dtype, same max context, same memory fraction. Otherwise you're comparing configs, not engines.
> - Warm up first (CUDA graphs and compile happen on early requests).
> - Fixed output length (`ignore_eos`) or you're benchmarking how chatty the sampler was.
> - A 1.5B model on a big GPU is **overhead-bound** (§8): differences here are mostly scheduler/Python overhead and won't transfer to a 70B model. Re-run with the model you'll actually ship.

---
# Part F: The hardware, measured

The slides quote spec-sheet numbers. Here we measure **your** GPU's two numbers that matter (memory bandwidth and matmul throughput) and derive its ops:byte ratio, the ridge point of the roofline.

| GPU | Memory | Bandwidth | BF16 dense TFLOPS | ops:byte | Notes |
|---|---|---|---|---|---|
| T4 | 16 GB GDDR6 | 0.32 TB/s | 65 (FP16) | ~200 | No BF16 → use `--dtype half`; no FlashAttention 2 |
| L4 | 24 GB GDDR6 | 0.30 TB/s | 121 | ~400 | FP8 capable (Ada) |
| A100 80GB | HBM2e | 2.0 TB/s | 312 | ~156 | No FP8 |
| H100 SXM | 80 GB HBM3 | 3.35 TB/s | 989 | ~295 | FP8; NVLink 900 GB/s |
| H200 | 141 GB HBM3e | 4.8 TB/s | 989 | ~206 | Same compute as H100, more/faster memory: a decode GPU |
| B200 | 192 GB HBM3e | ~8 TB/s | ~2,250 | ~280 | FP4 tensor cores; NVLink 1.8 TB/s |
| MI300X | 192 GB HBM3 | 5.3 TB/s | ~1,300 | ~245 | ROCm; vLLM & SGLang both support it |
| RTX 4090 | 24 GB GDDR6X | 1.0 TB/s | ~165 | ~165 | No NVLink → TP over PCIe is slow |
| RTX 5090 | 32 GB GDDR7 | 1.8 TB/s | ~210 | ~117 | FP4 (consumer Blackwell, sm_120) |
| Mac M3 Ultra | up to 512 GB unified | 0.82 TB/s | — | — | Huge memory, modest bandwidth: big MoE fits, decode is slowish |
| DGX Spark (GB10) | 128 GB unified | 0.27 TB/s | — | — | Fits 120B-class MoE; decode bandwidth-limited |

Spec-sheet numbers; vendor "sparse" TFLOPS are 2× these and rarely relevant to LLMs.

## 30. Inspect and micro-benchmark your GPU

In [ ]:
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cc = f"{p.major}.{p.minor}"
    print(f"{p.name} | {p.total_memory/1e9:.0f} GB | {p.multi_processor_count} SMs | compute capability sm_{p.major}{p.minor}")
    print("BF16 native:", p.major >= 8, "| FP8 tensor cores:", (p.major, p.minor) >= (8, 9), "| FP4 tensor cores:", p.major >= 10)

    # 1) Memory bandwidth: copy a 2 GB buffer (read + write)
    x = torch.empty(2 * 1024**3 // 2, dtype=torch.bfloat16, device="cuda"); y = torch.empty_like(x)
    for _ in range(3): y.copy_(x)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): y.copy_(x)
    torch.cuda.synchronize(); bw = 2 * x.numel() * 2 * 20 / (time.perf_counter() - t0) / 1e12
    del x, y

    # 2) Matmul throughput: large BF16 GEMM (dtype half on pre-Ampere)
    dt = torch.bfloat16 if p.major >= 8 else torch.float16
    a = torch.randn(8192, 8192, dtype=dt, device="cuda"); b = torch.randn_like(a)
    for _ in range(3): a @ b
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): a @ b
    torch.cuda.synchronize(); tflops = 2 * 8192**3 * 20 / (time.perf_counter() - t0) / 1e12
    del a, b; torch.cuda.empty_cache()

    print(f"\nmeasured bandwidth ≈ {bw:.2f} TB/s | measured {dt} matmul ≈ {tflops:.0f} TFLOPS")
    print(f"→ your ridge point (ops:byte) ≈ {tflops/bw:.0f} FLOPs/byte")
    print(f"→ batch-1 decode ceiling for this 1.5B BF16 model ≈ {bw*1e12/(1.54e9*2):.0f} tok/s")

In [ ]:
# Interconnect: how GPUs talk to each other. NV# = NVLink, PIX/PHB/SYS = PCIe paths (slower → prefer PP or DP over TP)
!nvidia-smi topo -m 2>/dev/null || echo "single GPU / no nvidia-smi"

> Measured bandwidth usually lands at 80–90% of spec; matmul at 60–80% of spec. Those *achievable* numbers, not the spec sheet, are what your MBU and cost estimates should use.

### The memory hierarchy in one table (H100 SXM)

| Level | Size | Bandwidth | What lives there during inference |
|---|---|---|---|
| Registers + shared memory / L1 (per SM, 132 SMs) | ~256 KB per SM | far above HBM (on-chip) | FlashAttention tiles, GEMM tiles |
| L2 cache | 50 MB | several × HBM | Hot activations; too small for weights |
| HBM3 | 80 GB | 3.35 TB/s | **Weights + KV cache**: what decode streams every step |
| NVLink 4 (GPU↔GPU) | — | 900 GB/s | Tensor-parallel all-reduces, KV transfer |
| PCIe Gen5 x16 (GPU↔CPU) | — | ~64 GB/s | CPU offload, KV offload, loading weights |
| CPU DRAM | 100s of GB–TBs | ~0.1–0.5 TB/s | Offloaded experts/KV (llama.cpp `--n-cpu-moe`, LMCache) |
| NVMe SSD | TBs | ~7–14 GB/s | Weight files; cold KV tier; your cold-start time |

Each step down is roughly an order of magnitude slower. Every "offload" trick trades capacity for bandwidth on this ladder.

---
# Part G: Hosting big models locally: the edge cases

"Can I run model X on machine Y?" has a two-line answer: **does it fit**, and **how fast does it stream**. Both come from §3's arithmetic, with three twists for local hardware: MoE (memory ∝ total params, speed ∝ active params), offload (part of the model lives on a slower tier), and quantization formats (GGUF, AWQ, FP8, MXFP4).

## 31. A planner that reads any Hugging Face config *(no GPU needed)*

In [ ]:
from huggingface_hub import hf_hub_download, HfApi

def model_facts(repo):
    # Pull config.json + safetensors header; works for dense, GQA, MLA and MoE models.
    cfg = json.load(open(hf_hub_download(repo, "config.json")))
    cfg = cfg.get("text_config", cfg)                                       # multimodal wrappers
    try:
        params = sum(HfApi().get_safetensors_metadata(repo).parameter_count.values())
    except Exception:
        params = None
    L = cfg["num_hidden_layers"]
    if "kv_lora_rank" in cfg:                                               # MLA (DeepSeek-V2/V3, Kimi K2)
        kv_elems = L * (cfg["kv_lora_rank"] + cfg.get("qk_rope_head_dim", 64))
    else:
        hd = cfg.get("head_dim") or cfg["hidden_size"] // cfg["num_attention_heads"]
        kv_elems = 2 * L * cfg.get("num_key_value_heads", cfg["num_attention_heads"]) * hd
    experts = cfg.get("num_local_experts") or cfg.get("n_routed_experts") or cfg.get("num_experts")
    active = cfg.get("num_experts_per_tok") or cfg.get("num_experts_per_token")
    return {"repo": repo, "params_B": params / 1e9 if params else None, "layers": L,
            "kv_bytes_per_token_bf16": kv_elems * 2, "experts": experts, "experts_per_token": active,
            "sliding_window": cfg.get("sliding_window"), "max_pos": cfg.get("max_position_embeddings")}

for repo in ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen3-30B-A3B", "openai/gpt-oss-120b"]:
    try:
        f = model_facts(repo); print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in f.items()})
    except Exception as e:
        print(repo, "→", type(e).__name__, "(gated? run `huggingface-cli login`)")

In [ ]:
HW = {  # name: (fast memory GB, fast BW GB/s, slow memory GB, slow BW GB/s)  slow = system RAM for offload
    "RTX 4090 + 64GB DDR5":   (24, 1000, 64, 60),
    "RTX 5090 + 128GB DDR5":  (32, 1790, 128, 70),
    "2x RTX 4090 (PCIe)":     (48, 2000, 64, 60),
    "Mac M3 Ultra 512GB":     (512 * 0.75, 819, 0, 1),   # macOS lets the GPU wire ~75% by default
    "DGX Spark 128GB":        (128 * 0.9, 273, 0, 1),
    "1x H100 80GB":           (80, 3350, 0, 1),
    "8x H100 (TP=8)":         (640, 26800, 0, 1),
}
BYTES = {"BF16": 2.0, "FP8/Q8": 1.06, "Q4 (GGUF Q4_K_M / AWQ)": 0.56, "MXFP4": 0.53}

def plan(total_B, active_B, kv_kib_per_tok, ctx=32_768, users=1, fmt="Q4 (GGUF Q4_K_M / AWQ)", kv_bytes=1):
    w = total_B * BYTES[fmt]; kv = kv_kib_per_tok * 1024 * ctx * users * kv_bytes / 2 / 1e9
    rows = []
    for name, (fast, fbw, slow, sbw) in HW.items():
        need = w + kv + 1.5                                                 # +1.5 GB activations/runtime
        on_gpu = min(need, fast); spill = need - on_gpu
        fits = spill <= 0 or spill <= slow
        # bytes streamed per token: active weights (+ KV); spilled share comes from the slow tier
        act = active_B * BYTES[fmt]; frac_spill = max(0, spill - kv) / max(w, 1e-9)
        t = act * (1 - frac_spill) / fbw + act * frac_spill / sbw          # seconds per token (ceiling)
        rows.append({"hardware": name, "needs_GB": round(need), "offloaded_GB": round(max(spill, 0)),
                     "fits": "yes" if spill <= 0 else ("with offload" if fits else "NO"),
                     "decode_ceiling_tok_s": round(1 / t) if fits else None})
    return pd.DataFrame(rows)

print("Qwen3-235B-A22B, Q4, 32K context, FP8 KV:")
display(plan(235, 22, kv_kib_per_tok=188, fmt="Q4 (GGUF Q4_K_M / AWQ)"))
print("gpt-oss-120b, native MXFP4:")
display(plan(117, 5.1, kv_kib_per_tok=72, fmt="MXFP4"))
print("Llama-3.3-70B dense, Q4:")
display(plan(70, 70, kv_kib_per_tok=320))

> Two patterns jump out. **MoE + offload is surprisingly usable** because each token only touches the active experts. **Dense 70B on a single consumer GPU with offload is painful**: every token drags tens of GB across a ~60 GB/s RAM bus. These are *ceilings*; real numbers are typically 50–80% of them, and **prefill suffers far more from offload than decode** because the CPU is a poor matmul engine.

## 32. The local toolbox *(reading)*

| Tool | Best at | Key flags / commands |
|---|---|---|
| **llama.cpp** (`llama-server`, `llama-bench`) | Anything that doesn't fit in VRAM; GGUF quants; CPU + GPU split; Mac/AMD/Intel/Vulkan | `-ngl 99` (layers on GPU), `--n-cpu-moe N` (keep N layers' experts on CPU), `-ot "exps=CPU"`, `-c` ctx, `-fa on`, `--cache-type-k/v q8_0` |
| **Ollama / LM Studio** | Friendly wrappers around llama.cpp (and MLX in LM Studio) | `OLLAMA_FLASH_ATTENTION=1`, `OLLAMA_KV_CACHE_TYPE=q8_0`, `num_ctx` |
| **MLX / mlx-lm** | Apple Silicon, often faster than llama.cpp on Macs | `mlx_lm.generate`, `mlx_lm.server`, 4-bit `mlx-community/*` repos |
| **exo** | Pooling several Macs (Thunderbolt 5) for 1T-parameter MoEs | — |
| **KTransformers** | Big MoE with experts on CPU (AMX) + attention on GPU; strong prefill in hybrid mode | — |
| **vLLM / SGLang** | Multi-user serving on NVIDIA/AMD, even on 1–4 consumer GPUs | `--cpu-offload-gb`, `--enforce-eager`, `--max-model-len`, `--kv-cache-dtype fp8`, `--pipeline-parallel-size` |

In [ ]:
# llama.cpp reports prefill and decode separately: pp = prompt processing (prefill), tg = token generation (decode)
#   brew install llama.cpp        (macOS)   or download a release build from github.com/ggml-org/llama.cpp
LLAMA_CMDS = r'''
# 1) Measure prefill vs decode for a GGUF model, all layers on GPU
llama-bench -hf Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M -ngl 99 -p 512,4096 -n 128 -fa 1

# 2) Serve a big MoE that does NOT fit in VRAM: attention + shared layers on GPU, experts on CPU
llama-server -hf unsloth/Qwen3-235B-A22B-Instruct-2507-GGUF:Q4_K_M \
    -ngl 99 --n-cpu-moe 70 -c 32768 -fa on \
    --cache-type-k q8_0 --cache-type-v q8_0 --port 8080
#    lower --n-cpu-moe until you OOM, then step back up: every expert layer moved to GPU speeds up both phases

# 3) Same idea with an explicit tensor override (older builds)
llama-server -m model.gguf -ngl 99 -ot ".ffn_.*_exps.=CPU" -c 16384

# 4) Apple Silicon with MLX
pip install mlx-lm
mlx_lm.generate --model mlx-community/Qwen3-30B-A3B-4bit --prompt "Hello" --max-tokens 200
mlx_lm.server  --model mlx-community/Qwen3-30B-A3B-4bit --port 8080
'''
print(LLAMA_CMDS)
import shutil
if shutil.which("llama-bench"):
    !llama-bench -hf Qwen/Qwen2.5-1.5B-Instruct-GGUF:Q4_K_M -ngl 99 -p 512 -n 128

> Every `llama-server` endpoint is OpenAI-compatible too, so §26's `bench_openai("http://localhost:8080", "any")` benchmarks it with the same harness.

## 33. vLLM when the model *almost* fits

Three levers, cheapest first: shrink the KV cache (`--max-model-len`, FP8 KV), skip CUDA-graph memory (`--enforce-eager`), then spill weights to CPU (`--cpu-offload-gb`, slow: every step crosses PCIe).

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL, max_model_len=2048, gpu_memory_utilization=0.5,
          kv_cache_dtype="fp8", enforce_eager=True,
          cpu_offload_gb=1)          # pretend the GPU is tiny: 1 GB of weights lives in CPU RAM
t0 = time.perf_counter()
o = llm.generate(["Explain CPU offload in one paragraph."], SamplingParams(max_tokens=128, temperature=0))
n = len(o[0].outputs[0].token_ids); print(o[0].outputs[0].text[:300])
print(f"\n{n / (time.perf_counter() - t0):.0f} tok/s with offload. Compare with §8's number: that gap is PCIe.")
shutdown(llm)

## 34. Edge-case checklist: errors you will hit *(reading)*

| Symptom / error | Cause | Fix |
|---|---|---|
| `CUDA out of memory` at startup | Memory fraction too high, or another process on the GPU | `nvidia-smi` to find it; lower `--gpu-memory-utilization` / `--mem-fraction-static` |
| "max_model_len … larger than the maximum number of tokens that can be stored in KV cache" | Model's default context (often 128K) doesn't fit | `--max-model-len 32768`, FP8 KV cache, or more GPUs |
| `Bfloat16 is only supported on GPUs with compute capability ≥ 8.0` | T4 / V100 / older | `--dtype half` |
| FP8 / NVFP4 checkpoint won't load | FP8 needs sm_89+ (Ada/Hopper); FP4 needs Blackwell | Use AWQ/GPTQ INT4 or the BF16 checkpoint |
| `trust_remote_code` error | New architecture with custom modelling code | Add `--trust-remote-code` *after reading the code*, or upgrade the engine |
| Garbage or repeated output | Wrong chat template / tokenizer, or base model instead of instruct | Use `llm.chat` / chat endpoint; pass `--chat-template`; check the repo is `-Instruct` |
| Multi-GPU hangs on consumer cards | NCCL P2P over PCIe is broken on some boards | `NCCL_P2P_DISABLE=1`; prefer `--pipeline-parallel-size` over TP without NVLink |
| Docker multi-GPU crash (`bus error`, shm) | Tiny `/dev/shm` | `--ipc=host` or `--shm-size=16g` |
| 401 / 403 downloading weights | Gated model (Llama, Gemma) | Accept licence on HF; `huggingface-cli login` or `HF_TOKEN` |
| Download takes forever / fills disk | Pulling every quant in a GGUF repo | `huggingface-cli download repo --include "*Q4_K_M*"`; `HF_HUB_ENABLE_HF_TRANSFER=1` |
| First request very slow | Compile + CUDA graph capture + cold page cache | Warm-up requests; persist compile cache; keep weights on local NVMe |
| Mac: model "fits" but crashes | GPU can only wire ~75% of unified memory by default | `sudo sysctl iogpu.wired_limit_mb=<MB>` (at your own risk) |
| Long prompts slow with MoE offload | Prefill runs on CPU for offloaded experts | More expert layers on GPU, bigger `-ub`/`-b` batch, or KTransformers |
| Vision model OOMs on images | Encoder activations + many image tokens | Limit `--limit-mm-per-prompt`, lower image resolution, smaller `max_num_seqs` |
| Throughput collapses at long context | KV cache full → preemption / recompute | Watch `num_preemptions`; FP8 KV; fewer concurrent seqs |

## 35. What practitioners report (treat as anecdotes, not benchmarks) *(reading)*

| Setup | Reported result | Source |
|---|---|---|
| 4× Mac Studio M3 Ultra (exo, Thunderbolt 5), Kimi K2 Thinking (~1T MoE) | ~25–28 tok/s decode, <500 W | Jeff Geerling via [heise](https://heise.de/-11085716) |
| 12 GB GPU, Qwen3 MoE, raising `--n-cpu-moe` tuning | Decode 17 → 34 tok/s after re-tuning offload | [r/LocalLLaMA summary](https://aiweekly.co/alerts/qwen3-moe-cpu-expert-offload-doubles-decode-speed-on-12gb-vram) |
| Experts on CPU vs GPU, 26B MoE | Prefill ~5.3× slower, decode ~2.4× slower | [ai-muninn](https://ai-muninn.com/en/blog/run-119b-moe-on-3x-2080ti) |
| M5 MacBook Pro, Qwen3.5-9B | llama.cpp 22, MLX 25, uzu 92 tok/s | [Avi Chawla on X](https://x.com/_avichawla/status/2115010493899502058) |
| Llama 4 Maverick, hybrid CPU/GPU | Decode 25 tok/s but prefill only 18 tok/s | r/LocalLLaMA thread |

The pattern across all of them: **local decode is a memory-bandwidth story, local prefill is a compute story**, and offload hurts prefill most.

---
# Part H: Seeing it at the kernel level, and using it as a data scientist

Four experiments that the popular write-ups (kipply, Horace He, the Hugging Face continuous-batching post) describe in prose. Here you measure them.

## 36. A measured roofline: why batching is free until it isn't

A decode step is mostly `activations @ weights`, with the activation matrix having one row per sequence in the batch. Time a single 8192×8192 BF16 weight multiply as the batch grows. While memory-bound, the time stays flat (you're paying to read the weights anyway). Past the ridge point it starts climbing.

In [ ]:
if torch.cuda.is_available():
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    Wt = torch.randn(8192, 8192, dtype=dt, device="cuda")
    rows = []
    for B in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
        x = torch.randn(B, 8192, dtype=dt, device="cuda")
        for _ in range(5): x @ Wt
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(50): x @ Wt
        torch.cuda.synchronize(); t = (time.perf_counter() - t0) / 50
        rows.append({"batch": B, "time_us": t * 1e6, "TFLOPS": 2 * B * 8192 * 8192 / t / 1e12})
    roof = pd.DataFrame(rows); display(roof.round(1))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(roof.batch, roof.time_us, "o-"); a1.set(xscale="log", xlabel="batch (rows)", ylabel="µs per matmul", title="Flat = memory-bound: extra rows are free")
    a2.plot(roof.batch, roof.TFLOPS, "o-"); a2.set(xscale="log", yscale="log", xlabel="batch (rows)", ylabel="achieved TFLOPS", title="The roofline, measured")
    plt.tight_layout(); plt.show()
    del Wt, x; torch.cuda.empty_cache()

> The batch size where the time starts rising is your GPU's real ridge point, close to the ops:byte ratio from §30. Below it, a decode step serving 64 users costs about the same as one serving 1 user. That single plot is the economic case for continuous batching.

## 37. FlashAttention vs naive attention: memory is the story

Naive attention materialises the full N×N score matrix in HBM. FlashAttention tiles it through on-chip SRAM, so peak memory grows linearly with sequence length instead of quadratically.

In [ ]:
import torch.nn.functional as F
if torch.cuda.is_available():
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    def naive(q, k, v):
        s = (q @ k.transpose(-1, -2)) / q.shape[-1] ** 0.5          # [B, H, N, N] lives in HBM
        s = s + torch.triu(torch.full_like(s[0, 0], float("-inf")), 1)
        return torch.softmax(s, -1) @ v
    rows = []
    for N in [1024, 2048, 4096, 8192, 16384]:
        q, k, v = (torch.randn(1, 16, N, 128, dtype=dt, device="cuda") for _ in range(3))
        for name, fn in [("naive", naive), ("fused (SDPA/flash)", lambda q, k, v: F.scaled_dot_product_attention(q, k, v, is_causal=True))]:
            try:
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); base = torch.cuda.memory_allocated()
                fn(q, k, v); torch.cuda.synchronize(); t0 = time.perf_counter()
                for _ in range(5): fn(q, k, v)
                torch.cuda.synchronize()
                rows.append({"seq_len": N, "kernel": name, "ms": (time.perf_counter() - t0) / 5 * 1e3,
                             "peak_extra_MB": (torch.cuda.max_memory_allocated() - base) / 1e6})
            except torch.OutOfMemoryError:
                rows.append({"seq_len": N, "kernel": name, "ms": None, "peak_extra_MB": "OOM"})
        del q, k, v
    display(pd.DataFrame(rows).pivot(index="seq_len", columns="kernel").round(1))

> Doubling the sequence length roughly quadruples the naive kernel's extra memory and leaves the fused kernel's nearly flat. Same maths, same result (FlashAttention is exact), very different traffic to HBM.

## 38. Chunked prefill, live: one long prompt vs everyone else's tokens

Sixteen users are streaming answers. Halfway through, one request arrives with a ~3,500-token prompt. We record the **longest gap between tokens** the streaming users see (the stall) and the long request's TTFT, for two per-step token budgets.

In [ ]:
async def stall_test(budget):
    srv = launch(["vllm", "serve", MODEL, "--port", str(PORT), "--max-model-len", str(MAX_LEN),
                  "--gpu-memory-utilization", "0.80", "--max-num-batched-tokens", str(budget),
                  "--no-enable-prefix-caching"], "vllm_chunk.log", f"{BASE}/health")
    ac = AsyncOpenAI(base_url=f"{BASE}/v1", api_key="none")
    async def streamer(i):
        gaps, last = [], None
        s = await ac.chat.completions.create(model=MODEL, max_tokens=400, stream=True, extra_body={"ignore_eos": True},
                                             messages=[{"role": "user", "content": f"Count slowly, user {i}."}])
        async for c in s:
            now = time.perf_counter()
            if last: gaps.append(now - last)
            last = now
        return max(gaps)
    async def long_one():
        await asyncio.sleep(1.0)
        t0 = time.perf_counter()
        s = await ac.chat.completions.create(model=MODEL, max_tokens=8, stream=True,
                                             messages=[{"role": "user", "content": "word " * 3500 + "\nSummarise."}])
        async for c in s: return time.perf_counter() - t0
    res = await asyncio.gather(*[streamer(i) for i in range(16)], long_one())
    srv.terminate(); srv.wait(60)
    return {"max_num_batched_tokens": budget, "worst_stall_ms": max(res[:-1]) * 1e3, "long_prompt_ttft_ms": res[-1] * 1e3}

display(pd.DataFrame([await stall_test(512), await stall_test(8192)]).round(0))

> A small budget splits the long prefill across many steps: the streaming users barely notice, but the long request waits longer for its first token. A big budget does the opposite. There is no free setting, only a choice of which user pays. Disaggregation (§20) removes the trade-off by giving prefill its own GPUs.

## 39. The data-scientist workloads: embeddings and zero-shot labelling at scale

Two jobs that show up in every DS team and run far faster through an inference engine than through a `for` loop over `transformers`.

In [ ]:
from vllm import LLM, SamplingParams

# 1) Embeddings for 2,000 texts with a pooling model
texts = [f"Customer {i} says the delivery was {'late' if i % 3 else 'on time'} and the item was {'damaged' if i % 5 == 0 else 'fine'}." for i in range(2000)]
try:
    emb_llm = LLM(model="BAAI/bge-small-en-v1.5", runner="pooling", gpu_memory_utilization=0.3)
except TypeError:                                    # older vLLM used task="embed"
    emb_llm = LLM(model="BAAI/bge-small-en-v1.5", task="embed", gpu_memory_utilization=0.3)
t0 = time.perf_counter(); out = emb_llm.embed(texts); dt_ = time.perf_counter() - t0
print(f"{len(texts)} embeddings of dim {len(out[0].outputs.embedding)} in {dt_:.1f}s → {len(texts)/dt_:.0f} texts/s")
shutdown(emb_llm)

In [ ]:
# 2) Zero-shot labelling with a constrained label set, plus a confidence score from logprobs
llm = LLM(model=MODEL, max_model_len=1024, gpu_memory_utilization=0.6)
LABELS = ["late_delivery", "damaged_item", "positive"]
try:
    from vllm.sampling_params import StructuredOutputsParams
    sp = SamplingParams(temperature=0, max_tokens=8, logprobs=1, structured_outputs=StructuredOutputsParams(choice=LABELS))
except ImportError:                                  # older vLLM name
    from vllm.sampling_params import GuidedDecodingParams
    sp = SamplingParams(temperature=0, max_tokens=8, logprobs=1, guided_decoding=GuidedDecodingParams(choice=LABELS))
msgs = [[{"role": "user", "content": f"Label this review as one of {LABELS}.\nReview: {t}"}] for t in texts[:1000]]
t0 = time.perf_counter(); outs = llm.chat(msgs, sp); dt_ = time.perf_counter() - t0
import math
rows = [{"text": t[:60], "label": o.outputs[0].text,
         "confidence": math.exp(sum(next(iter(lp.values())).logprob for lp in o.outputs[0].logprobs))} for t, o in zip(texts, outs)]
display(pd.DataFrame(rows).head(8))
print(f"1,000 labels in {dt_:.1f}s. Every output is guaranteed to be one of {LABELS}.")
shutdown(llm)

> Constrained choices remove the parsing step and the "model answered in a sentence" failure mode. The logprob-based confidence lets you route low-confidence rows to a bigger model or a human: a cheap way to cut labelling cost.

## 40. Profiling: where does a step actually go? *(reading)*

When a number surprises you, look at the kernel timeline. vLLM wraps the PyTorch profiler:

```python
import os; os.environ["VLLM_TORCH_PROFILER_DIR"] = "./vllm_profile"   # set before creating the LLM
llm = LLM(model=MODEL)
llm.start_profile()
llm.generate(["Hello"] * 32, SamplingParams(max_tokens=64))
llm.stop_profile()          # open the trace in https://ui.perfetto.dev
```

What to look for: long gaps between kernels (overhead regime, so check CUDA graphs are on), attention vs GEMM share (long context → attention-dominated), and NCCL all-reduce time with tensor parallelism. For whole-system views use Nsight Systems: `nsys profile -o trace vllm serve …`.

> Same GPU, same price: **utilization is the cost lever**. Batching, prefix caching and quantization all show up here as a smaller number.

---

## Exercises for the team

1. **Find your goodput.** Using step 11, find the highest concurrency where per-user speed ≥ 30 tok/s. What's the system throughput there?
2. **Break prefix caching.** Add `f"Current time: {time.time()}"` to the *top* of the system prompt in step 12. What happens to warm TTFT? Now move it to the bottom.
3. **Long prompts.** Re-run `vllm bench serve` with `--random-input-len 3000`. Which metric degrades most: TTFT or TPOT? Why?
4. **Spec decode at load.** Serve with `--speculative-config '{"method":"ngram","num_speculative_tokens":4,"prompt_lookup_max":4}'` and repeat the concurrency sweep. At what concurrency does the benefit vanish?
5. **Capacity planning.** For Llama-3.1-8B on H100 with 8K contexts, use the step 3 functions to estimate max concurrency in BF16 vs FP8 KV cache.

## Further reading
- **Inference Engineering**, Philip Kiely (Baseten). Free online: [baseten.co/inference-engineering](https://www.baseten.co/inference-engineering/)
- [vLLM documentation](https://docs.vllm.ai) · [vLLM blog](https://vllm.ai/blog)
- **Inside vLLM: Anatomy of a High-Throughput LLM Inference System**, Aleksa Gordić
- Kwon et al., *PagedAttention* (SOSP '23) · Yu et al., *Orca* (OSDI '22) · Zhong et al., *DistServe* (OSDI '24)
- Dao et al., *FlashAttention* 1–3 · Leviathan et al., *Speculative Decoding* (2023) · Li et al., *EAGLE-3*
- [Databricks: LLM Inference Performance Engineering](https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices): metrics, MBU, honest benchmarking
- [kipply: Transformer Inference Arithmetic](https://kipp.ly/transformer-inference-arithmetic/): the back-of-envelope maths behind §3 and §36
- [Horace He: Making Deep Learning Go Brrrr](https://horace.io/brrr_intro.html): compute vs bandwidth vs overhead
- [P. Lienhart: LLM Inference Series](https://medium.com/@plienhar/llm-inference-series-3-kv-caching-unveiled-048152e461c8) (Medium): the KV cache, step by step
- [Hugging Face: Continuous batching from first principles](https://huggingface.co/blog/continuous%5Fbatching)
- [Modal: LLM Engine Almanac](https://modal.com/llm-almanac): vLLM vs SGLang vs TensorRT-LLM, measured
- [SGLang docs](https://docs.sglang.ai) · [TensorRT-LLM docs](https://nvidia.github.io/TensorRT-LLM/) · [llama.cpp](https://github.com/ggml-org/llama.cpp) · [mlx-lm](https://github.com/ml-explore/mlx-lm)
- [Google: How to Scale Your Model, inference chapter](https://jax-ml.github.io/scaling-book/inference)

In [ ]:
# Final cleanup: stop anything still running
try:
    if server.poll() is None: server.terminate()
except NameError:
    pass
gc.collect(); print(gpu_mem())